In [ ]:
import pandas as pd
import sys
sys.path.append('..')
import util

rppa, rna_raw, clin = util.loadTCGAData()
egfrviii = util.loadEGFRvIIIData()

raw_data = [rppa, rna_raw, egfrviii]

for df in raw_data:
    util.printDataShape(df)

The first part of the ipynb file is completely based on Alhem's initial exploratory analysis. 

In [ ]:
for df in raw_data:
    display(df.head())

In [ ]:
# Data cleaning and preprocessing

import re
import numpy as np

# 1) choose the gene column (prefer Hugo_Symbol if present)
gene_col = None
for c in rna_raw.columns:
    if str(c).lower() in ['hugosymbol', 'gene', 'gene_name', 'symbol', 'genes']:
        gene_col = c
        break
if gene_col is None: 
    gene_col = rna_raw.columns[0]
print("the detected gene column is:", gene_col)
# 2) detect TCGA sample columns (TCGA-XX-XXXX-YY...)
tcga_pat = re.compile(r"^TCGA-[A-Z0-9]{2}-[A-Z0-9]{4}-\d{2}", re.IGNORECASE) # this is to detect TCGA sample columns with regex
sample_cols = [c for c in rna_raw.columns if tcga_pat.match(str(c))]

# 3) slim to gene + sample columns, drop NA genes, group duplicate genes by mean
rna_slim = rna_raw[[gene_col]+sample_cols].copy() # this is to create a slim version of rna_raw with only gene and sample columns
rna_slim = rna_slim.dropna(subset=[gene_col]) 
rna_slim = rna_slim.groupby(gene_col, as_index = True).mean(numeric_only  = True) # this is to group duplicate genes by mean

# 4) transpose to samples * genes
rna = rna_slim.T
rna.index.name = "Sample_ID" # set index name, displayed in rna.head()

# 5) harmonize RPPA columns and compute intersection
rppa.columns = pd.Index([c.strip() for c in rppa.columns])
# # sorted() computes the intersection of sample names
common_samples = sorted(set(rppa.columns) & set(rna.index)) # set(rppa.columns) is the set of RPPA sample names, set(rna.index) is the set of RNA sample names (row indices after transpose)

print("Parsed RNA matrix (samples * genes):" , rna.shape)
print("RPPA matrix (features * samples):", rppa.shape)
print("Common samples:", len(common_samples))
print("common_samples example:", common_samples[:5])

In [ ]:
# 6) quick QC counts (before any normalization)
rna_na_total = int(rna.isna().sum().sum()) # total NA cells in rna
rppa_na_rows = int((rppa.isna().sum(axis=1) > 0).sum()) # number of RPPA rows with any NA
rna_zero_var = int((rna.std(axis=0) == 0).sum()) # number of RNA genes with zero variance
rppa_zero_var = int((rppa.std(axis=1) == 0).sum()) # the number of RPPA features with zero variance

print("\nQC:")
print("  RNA total NA cells:", rna_na_total)
print("  RPPA rows with any NA:", rppa_na_rows)
print("  RNA genes with zero variance:", rna_zero_var)
print("  RPPA features with zero variance:", rppa_zero_var)

# 7) show a tiny peek so we can visually confirm structure
print("\nRNA (5 samples × 5 genes):")
display(rna.iloc[:5, :5])

print("\nRPPA (5 features × 5 samples):")
display(rppa.iloc[:5, :5])

In [ ]:
# Normalize (z-score) and clean
# RPPA: z-score each feature (row)
rppa_z = rppa.sub(rppa.mean(axis = 1), axis = 0) # rppa.sub() subtracts the mean of each row from each element in that row)
rppa_z = rppa_z.div(rppa.std(axis = 1).replace(0, np.nan), axis = 0) # rppa.div() divides each element in a row by the std of that row; replace(0, np.nan) avoids division by zero

# RNA: z-score each gene (column)
rna_z = (rna - rna.mean(axis = 0)) / rna.std(axis = 0).replace(0, np.nan)

# Subset to common samples
rppa_z = rppa_z[common_samples]
rna_z  = rna_z.loc[common_samples]

print("RPPA_z: ", rppa_z.shape)
print("RNA_z: ", rna_z.shape)

print("\n Missing values check:")
print(" RPPA_z any NaN:", rppa_z.isna().any().any())
print(" RNA_z any NaN:", rna_z.isna().any().any())


In [ ]:
#Strict numeric clean, impute, drop-zero-var, z-score, PCA (RPPA)
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Start from RPPA restricted to common samples
R = rppa.loc[:, common_samples].copy()

# 1) Force numeric (coerce non-numeric to NaN)
R = R.apply(pd.to_numeric, errors="coerce")

# 2) Drop rows that are entirely NaN
n_allna_before = int((R.isna().all(axis=1)).sum())
R = R.loc[~R.isna().all(axis=1)]
print("Rows dropped (all-NaN):", n_allna_before)

# 3) Impute remaining NaNs per row with the row median
R_imp = R.apply(lambda row: row.fillna(row.median()), axis=1)

# 4) Drop rows that are constant after imputation (std == 0)
row_std = R_imp.std(axis=1)
n_const = int((row_std == 0).sum())
R_imp = R_imp.loc[row_std > 0]
print("Rows dropped (zero-variance):", n_const)

# 5) Z-score per row
R_z = R_imp.sub(R_imp.mean(axis=1), axis=0)
R_z = R_z.div(R_imp.std(axis=1), axis=0)



# 6) Final guard: replace any residual NaN with 0
had_nan = bool(R_z.isna().any().any())
R_z = R_z.fillna(0.0)
print("Had NaN after z-score (pre-fill)?", had_nan)
print("Any NaN now?:", bool(R_z.isna().any().any()))





In [ ]:
# 7) PCA (samples x features)
X = R_z.T.values
pca_rppa = PCA(n_components=5, random_state=0).fit(X)
scores = pca_rppa.transform(X)

# Scree
plt.figure(figsize=(5,3))
plt.plot(range(1,6), pca_rppa.explained_variance_ratio_, marker='o')
plt.xlabel("PC")
plt.ylabel("Variance explained") # the ratio of sigma_k^2/sum(sigma_j^2)
plt.title("RPPA PCA — Scree")
plt.show()

# PC1 vs PC2
plt.figure(figsize=(5,4))
plt.scatter(scores[:,0], scores[:,1], alpha=0.7)
plt.xlabel(f"PC1 ({pca_rppa.explained_variance_ratio_[0]*100:.1f}% var)")
plt.ylabel(f"PC2 ({pca_rppa.explained_variance_ratio_[1]*100:.1f}% var)")
plt.title("RPPA PCA — samples")
plt.show()

# Loadings for PC1 (feature contributions)
pc1_load = pd.Series(pca_rppa.components_[0], index=R_z.index).sort_values(ascending=False)
print("Top 5 PC1 loadings:")
print(pc1_load.head(5))
print("\nBottom 5 PC1 loadings:")
print(pc1_load.tail(5))

In [ ]:
pca_rppa.components_

In [ ]:
#PCA on RNA (restrict to high-variance genes)

# Subset to common samples
rna_sub = rna.loc[common_samples] # extract in the rows corresponding to common samples
# rna_sub = rna_z
# Compute variance per gene
gene_var = rna_sub.var(axis=0)
top_genes = gene_var.sort_values(ascending=False).head(2000).index # top 2000 most variable genes
rna_hv = rna_sub[top_genes] # subset to these genes

# Z-score each gene (re-normalize within this subset)
rna_hv_z = (rna_hv - rna_hv.mean(axis=0)) / rna_hv.std(axis=0).replace(0, np.nan)
rna_hv_z = rna_hv_z.fillna(0.0)

print("RNA high-variance subset:", rna_hv_z.shape)

# PCA
X_rna = rna_hv_z.values
pca_rna = PCA(n_components=10, random_state=0).fit(X_rna)
rna_embed = pca_rna.transform(X_rna)

# Scree
plt.figure(figsize=(5,3))
plt.plot(range(1,11), pca_rna.explained_variance_ratio_, marker='o')
plt.xlabel("PC")
plt.ylabel("Variance explained")
plt.title("RNA PCA — Scree")
plt.show()

# PC1 vs PC2 scatter
plt.figure(figsize=(5,4))
plt.scatter(rna_embed[:,0], rna_embed[:,1], alpha=0.7)
plt.xlabel(f"PC1 ({pca_rna.explained_variance_ratio_[0]*100:.1f}% var)")
plt.ylabel(f"PC2 ({pca_rna.explained_variance_ratio_[1]*100:.1f}% var)")
plt.title("RNA PCA — samples (top 2000 variable genes)")
plt.show()

# Top/bottom loadings for PC1
pc1_load_rna = pd.Series(pca_rna.components_[0], index=rna_hv_z.columns).sort_values(ascending=False)
print("Top 5 PC1 loadings (RNA):")
print(pc1_load_rna.head(5))
print("\nBottom 5 PC1 loadings (RNA):")
print(pc1_load_rna.tail(5), "\n")

# Top/bottom loadings for PC2
pc2_load_rna = pd.Series(pca_rna.components_[1], index=rna_hv_z.columns).sort_values(ascending=False)
print("Top 5 PC2 loadings (RNA):")
print(pc2_load_rna.head(5))
print("\nBottom 5 PC2 loadings (RNA):")
print(pc2_load_rna.tail(5), "\n")

In [ ]:
#Side-by-side PCA plots for RPPA vs RNA

fig, axes = plt.subplots(1, 2, figsize=(10,4))

# RPPA scatter
axes[0].scatter(scores[:,0], scores[:,1], alpha=0.7)
axes[0].set_xlabel(f"PC1 ({pca_rppa.explained_variance_ratio_[0]*100:.1f}% var)")
axes[0].set_ylabel(f"PC2 ({pca_rppa.explained_variance_ratio_[1]*100:.1f}% var)")
axes[0].set_title("RPPA PCA (proteins/phospho)")

# RNA scatter
axes[1].scatter(rna_embed[:,0], rna_embed[:,1], alpha=0.7, color="darkorange")
axes[1].set_xlabel(f"PC1 ({pca_rna.explained_variance_ratio_[0]*100:.1f}% var)")
axes[1].set_ylabel(f"PC2 ({pca_rna.explained_variance_ratio_[1]*100:.1f}% var)")
axes[1].set_title("RNA PCA (top 2000 variable genes)")

plt.tight_layout()
plt.show()

In [ ]:
#Compare sample distances across RNA vs RPPA

from sklearn.metrics import pairwise_distances
import seaborn as sns

# Use PCA scores (PC1–PC5) for RNA and RPPA
p = 1
rna_dist = pairwise_distances(rna_embed[:, :5], metric='minkowski', p=p) ** p 
rppa_dist = pairwise_distances(scores[:, :5], metric='minkowski', p=p) ** p

# Flatten upper triangles for correlation
mask = np.triu(np.ones(rna_dist.shape), k=1).astype(bool)
rna_flat = rna_dist[mask]
rppa_flat = rppa_dist[mask]

# Correlation between distance structures
from scipy.stats import spearmanr
corr, pval = spearmanr(rna_flat, rppa_flat)

print(f"RNA–RPPA sample distance correlation: r={corr:.3f}, p={pval:.2e}")

# Scatterplot of distances
plt.figure(figsize=(5,5))
plt.scatter(rna_flat, rppa_flat, alpha=0.3)
plt.xlabel("RNA sample distance (PC1–5)")
plt.ylabel("RPPA sample distance (PC1–5)")
plt.title(f"Sample-level concordance RNA vs RPPA (Spearman r={corr:.2f})")
plt.show()

# Heatmap comparison
fig, axes = plt.subplots(1,2, figsize=(10,4))
sns.heatmap(rna_dist, cmap="viridis", ax=axes[0])
axes[0].set_title("RNA sample distances (PC1–5)")
sns.heatmap(rppa_dist, cmap="viridis", ax=axes[1])
axes[1].set_title("RPPA sample distances (PC1–5)")
plt.show()



In [ ]:
print("The shape of rna_dist is: ", rna_dist.shape)

In [ ]:
#Feature-level correlation (RNA vs RPPA)

from scipy.stats import pearsonr

# Match features: extract gene names from RPPA (before the '|')
rppa_genes = rppa.index.to_series().str.split("|").str[0]
common_genes = rppa_genes[rppa_genes.isin(rna.columns)].unique()

results = []
for g in common_genes:
    # RNA vector
    rna_vec = rna_z[g].loc[common_samples]
    # All RPPA rows mapping to this gene
    rows = rppa.index[rppa.index.str.startswith(g+"|")]
    for row in rows:
        rppa_vec = rppa_z.loc[row, common_samples]
        if rna_vec.std() > 0 and rppa_vec.std() > 0:
            r, p = pearsonr(rna_vec, rppa_vec)
            results.append((g, row, r, p, len(common_samples)))

res_df = pd.DataFrame(results, columns=["Gene","RPPA_feature","PearsonR","Pval","N"])
res_df["log10P"] = -np.log10(res_df["Pval"])

print("Correlated features:", res_df.shape)
print(res_df.sort_values("PearsonR", ascending=False).head(10))

# Volcano plot
plt.figure(figsize=(6,5))
plt.scatter(res_df["PearsonR"], res_df["log10P"], alpha=0.6)
plt.axvline(0, color="grey", linestyle="--")
plt.xlabel("Pearson correlation (RNA vs RPPA)")
plt.ylabel("-log10(p-value)")
plt.title("RNA–RPPA feature-level correlations")
plt.show()


In [ ]:
#Heatmaps of top concordant and discordant features

# Sort by correlation
res_sorted = res_df.sort_values("PearsonR", ascending=False)

# Top 10 concordant
top10 = res_sorted.head(10).copy()
# Bottom 10 discordant (lowest correlations)
bot10 = res_sorted.tail(10).copy()

def _prep_rna_mat(genes):
    # keep only genes that exist in rna_z
    genes = [g for g in genes if g in rna_z.columns]
    M = rna_z.loc[common_samples, genes]          # samples × genes
    # drop duplicate columns if any
    M = M.loc[:, ~M.columns.duplicated()]
    # drop columns/rows that are all NaN or zero variance
    keep_cols = (M.notna().any(axis=0)) & (M.std(axis=0, ddof=1) > 0)
    M = M.loc[:, keep_cols]
    keep_rows = (M.notna().any(axis=1)) & (M.std(axis=1, ddof=1) > 0)
    M = M.loc[keep_rows]
    return M

def _prep_rppa_mat(features):
    feats = [f for f in features if f in rppa_z.index]
    M = rppa_z.loc[feats, common_samples].T        # samples × features
    # drop duplicate columns if any
    M = M.loc[:, ~M.columns.duplicated()]
    # drop columns/rows that are all NaN or zero variance
    keep_cols = (M.notna().any(axis=0)) & (M.std(axis=0, ddof=1) > 0)
    M = M.loc[:, keep_cols]
    keep_rows = (M.notna().any(axis=1)) & (M.std(axis=1, ddof=1) > 0)
    M = M.loc[keep_rows]
    return M

def plot_heatmap_pair(subset, title_prefix, tag):
    genes = subset["Gene"].tolist()
    feats = subset["RPPA_feature"].tolist()

    M_rna  = _prep_rna_mat(genes)
    M_rppa = _prep_rppa_mat(feats)

    # If a matrix ends up empty, report and skip plotting to avoid blank figs
    if M_rna.shape[1] == 0 or M_rna.shape[0] == 0:
        print(f"[WARN] RNA matrix is empty for {title_prefix} (after cleaning).")
    else:
        g1 = sns.clustermap(
            M_rna, cmap="vlag", center=0, robust=True,
            figsize=(7,7), dendrogram_ratio=(.15,.15),
            cbar_pos=(0.02, .8, .03, .15)
        )
        g1.ax_heatmap.set_title(f"{title_prefix} (RNA)", pad=12)
        plt.savefig(f"heatmap_{tag}_RNA.png", dpi=150, bbox_inches="tight")
        plt.show()

    if M_rppa.shape[1] == 0 or M_rppa.shape[0] == 0:
        print(f"[WARN] RPPA matrix is empty for {title_prefix} (after cleaning).")
    else:
        g2 = sns.clustermap(
            M_rppa, cmap="vlag", center=0, robust=True,
            figsize=(7,7), dendrogram_ratio=(.15,.15),
            cbar_pos=(0.02, .8, .03, .15)
        )
        g2.ax_heatmap.set_title(f"{title_prefix} (RPPA)", pad=12)
        plt.savefig(f"heatmap_{tag}_RPPA.png", dpi=150, bbox_inches="tight")
        plt.show()

    # Return shapes so you can sanity-check
    print(f"{title_prefix} shapes — RNA {M_rna.shape}, RPPA {M_rppa.shape}")

print("Top 10 correlated genes/features:")
display(top10[["Gene","RPPA_feature","PearsonR"]])
plot_heatmap_pair(top10, "Top concordant features", "top_concordant")

print("Bottom 10 correlated genes/features:")
display(bot10[["Gene","RPPA_feature","PearsonR"]])
plot_heatmap_pair(bot10, "Bottom discordant features", "bottom_discordant")


Below are some exploratory analysis by Wenyan Luo. I tried ISOMAP and diffusion map embedding of the common samples onto 2D planes to get some sense of the data's shape. 

In [ ]:
# ISOMAP embedding
from sklearn.manifold import Isomap

# Fit ISOMAP to RPPA data
isomap = Isomap(n_components=2, n_neighbors=10)
X_rppa = R_z.T.values
isomap.fit(X_rppa)
isomap_embedding_rppa = isomap.transform(X_rppa)

print("The shape of rppa_z is:", X_rppa.shape)
print("The shape of isomap_embedding_rppa is:", isomap_embedding_rppa.shape)

# Fit ISOMAP to RNA data
isomap = Isomap(n_components=2, n_neighbors=10)
X_rna = rna_hv_z.values
isomap.fit(X_rna)
isomap_embedding_rna = isomap.transform(X_rna)

print("The shape of rna_hv_z is:", X_rna.shape)
print("The shape of isomap_embedding_rna is:", isomap_embedding_rna.shape)

In [ ]:
# define a color array based on sample index or a label
colors = np.linspace(0, 1, len(X_rppa))   # or e.g. colors = y if you have class labels

# Plot RPPA data and its ISOMAP embedding
# Plot original RPPA data
plt.figure(figsize=(8, 6))
plt.scatter(X_rppa[:, 0], X_rppa[:, 1], c=colors, cmap='Spectral')
plt.figure(figsize=(8, 6))
plt.scatter(isomap_embedding_rppa[:, 0], isomap_embedding_rppa[:, 1], c=colors, cmap='Spectral')

# Plot RNA data and its ISOMAP embedding
plt.figure(figsize=(8, 6))
plt.scatter(X_rna[:, 0], X_rna[:, 1], c=colors, cmap='Spectral')
plt.figure(figsize=(8, 6))
plt.scatter(isomap_embedding_rna[:, 0], isomap_embedding_rna[:, 1], c=colors, cmap='Spectral')



In [ ]:
# Example colors (you already have these)
colors = np.linspace(0, 1, len(X_rppa))

# --- Plot RPPA original vs. Isomap embedding with connecting lines ---
plt.figure(figsize=(8, 8))
plt.scatter(X_rppa[:, 0], X_rppa[:, 1], c="blue", cmap='Spectral', label='Original space', alpha=0.6)
plt.scatter(isomap_embedding_rppa[:, 0], isomap_embedding_rppa[:, 1],
            c="red", cmap='Spectral', marker='x', label='Isomap embedding', alpha=0.8)

# Draw lines connecting corresponding points
for i in range(0, len(X_rppa), 1):  # every 5th to avoid clutter; remove the step for all points
    plt.plot([X_rppa[i, 0], isomap_embedding_rppa[i, 0]],
             [X_rppa[i, 1], isomap_embedding_rppa[i, 1]],
             color='gray', alpha=0.3, linewidth=0.8)

# Add number labels for each data point
for i in range(len(X)):
    plt.text(isomap_embedding_rppa[i, 0],
             isomap_embedding_rppa[i, 1],
             str(i),                 # label = index number
             fontsize=8,
             ha='right', va='bottom')
plt.legend()
plt.title("Matched Line Plot: RPPA - Original vs Isomap Embedding")
plt.xlabel("X₁ / Isomap₁")
plt.ylabel("X₂ / Isomap₂")
plt.gca().set_aspect('equal', adjustable='box')
plt.show()


In [ ]:
# Example colors (you already have these)
colors = np.linspace(0, 1, len(X_rna))

# --- Plot RNA original vs. Isomap embedding with connecting lines ---
plt.figure(figsize=(8, 8))
plt.scatter(X_rna[:, 0], X_rna[:, 1], c="blue", cmap='Spectral', label='Original space', alpha=0.6)
plt.scatter(isomap_embedding_rna[:, 0], isomap_embedding_rna[:, 1],
            c="red", cmap='Spectral', marker='x', label='Isomap embedding', alpha=0.8)

# Draw lines connecting corresponding points
for i in range(0, len(X_rna), 1):  # every 5th to avoid clutter; remove the step for all points
    plt.plot([X_rna[i, 0], isomap_embedding_rna[i, 0]],
             [X_rna[i, 1], isomap_embedding_rna[i, 1]],
             color='gray', alpha=0.3, linewidth=0.8)
# Add number labels for each data point
for i in range(len(X_rna)):
    plt.text(isomap_embedding_rna[i, 0],
             isomap_embedding_rna[i, 1],
             str(i),                 # label = index number
             fontsize=8,
             ha='right', va='bottom')
plt.legend()
plt.title("Matched Line Plot: RNA - Original vs Isomap Embedding")
plt.xlabel("X₁ / Isomap₁")
plt.ylabel("X₂ / Isomap₂")
plt.gca().set_aspect('equal', adjustable='box')
plt.show()

In [ ]:
# Example colors
colors_rppa = np.linspace(0, 1, len(X_rppa))
colors_rna  = np.linspace(0, 1, len(X_rna))

# --- Create 1×2 subplot layout ---
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ===== (1) RPPA matched-line plot =====
ax = axes[0]
ax.scatter(X_rppa[:, 0], X_rppa[:, 1], c="blue", alpha=0.6, label='RPPA after z-score')
ax.scatter(isomap_embedding_rppa[:, 0], isomap_embedding_rppa[:, 1],
           c="red", marker='x', alpha=0.8, label='Isomap embedding')

for i in range(0, len(X_rppa), 1):
    ax.plot([X_rppa[i, 0], isomap_embedding_rppa[i, 0]],
            [X_rppa[i, 1], isomap_embedding_rppa[i, 1]],
            color='gray', alpha=0.3, linewidth=0.8)

# Number labels
for i in range(len(X_rppa)):
    ax.text(isomap_embedding_rppa[i, 0], isomap_embedding_rppa[i, 1],
            str(i), fontsize=10, ha='right', va='bottom')

ax.set_title("Matched Line Plot: RPPA - Original vs Isomap Embedding")
ax.set_xlabel("X₁ / Isomap₁")
ax.set_ylabel("X₂ / Isomap₂")
ax.legend()
ax.set_aspect('equal', adjustable='box')


# ===== (2) RNA matched-line plot =====
ax = axes[1]
ax.scatter(X_rna[:, 0], X_rna[:, 1], c="blue", alpha=0.6, label='RNA after z-score')
ax.scatter(isomap_embedding_rna[:, 0], isomap_embedding_rna[:, 1],
           c="red", marker='x', alpha=0.8, label='Isomap embedding')

for i in range(0, len(X_rna), 1):
    ax.plot([X_rna[i, 0], isomap_embedding_rna[i, 0]],
            [X_rna[i, 1], isomap_embedding_rna[i, 1]],
            color='gray', alpha=0.3, linewidth=0.8)

# Number labels
for i in range(len(X_rna)):
    ax.text(isomap_embedding_rna[i, 0], isomap_embedding_rna[i, 1],
            str(i), fontsize=10, ha='right', va='bottom')

ax.set_title("Matched Line Plot: RNA - Original vs Isomap Embedding")
ax.set_xlabel("X₁ / Isomap₁")
ax.set_ylabel("X₂ / Isomap₂")
ax.legend()
ax.set_aspect('equal', adjustable='box')

# --- Layout adjustment ---
plt.tight_layout()
plt.show()


In [ ]:
# Implement Diffusion Maps of the ISOMAP embedded data

def diffusion_map_fit(X, n_components=2, epsilon=None, alpha=1.0):
    X = np.asarray(X, dtype=float)
    n = X.shape[0]

    D2 = cdist(X, X, metric='sqeuclidean')
    if epsilon is None:
        epsilon = np.median(D2[D2 > 0])
    K = np.exp(-D2 / (epsilon + 1e-12))
    np.fill_diagonal(K, 1.0)

    d = K.sum(axis=1)
    d_alpha = np.power(d + 1e-15, -alpha)
    K_alpha = (d_alpha[:, None] * K) * d_alpha[None, :]

    q = K_alpha.sum(axis=1)
    P = K_alpha / (q[:, None] + 1e-15)

    k = min(n - 1, n_components + 1)
    evals, evecs = eigs(P.T, k=k, which='LM')
    
    order = np.argsort(evals.real)[::-1] # sort by descending eigenvalue
    evals = evals[order].real
    evecs = evecs[:, order].real

    return {'evecs': evecs, 'evals': evals, 'n_components': n_components}


def diffusion_map_transform(fitted_model, t):
    evecs = fitted_model['evecs']
    evals = fitted_model['evals']
    n_components = fitted_model['n_components']
    
    evecs_nt = evecs[:, 1:n_components+1]
    evals_nt = evals[1:n_components+1]
    
    Y = evecs_nt * (evals_nt ** t)
    return Y



# Diffusion Map of ISOMAP RPPA data and the original RPPA data


# Fit
model_isomap_rppa = diffusion_map_fit(isomap_embedding_rppa, n_components=2)
model_rppa   = diffusion_map_fit(X_rppa, n_components=2)
# --- Plot diffusion maps of ISOMAP RPPA and original RPPA data ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ===== (1) Left panel: ISOMAP RPPA embedding =====
ax = axes[0]
sc1 = ax.scatter(isomap_embedding_rppa[:, 0], isomap_embedding_rppa[:, 1],
                 c="blue", marker="x", s=20, alpha=0.6, label="ISOMAP RPPA")
for i in range(len(X_rppa)):
    ax.text(isomap_embedding_rppa[i, 0], isomap_embedding_rppa[i, 1],
            str(i), fontsize=8, ha='right', va='bottom')

ax.set_title("ISOMAP RPPA Embedding")
ax.set_aspect('equal')
ax.grid(alpha=0.3)
ax.legend(loc='best')

# ===== (2) Right panel: Diffusion Map comparison =====
t = 0.01
Diff_isomap_RPPA = diffusion_map_transform(model_isomap_rppa, t)
Diff_RPPA = diffusion_map_transform(model_rppa, t)

ax = axes[1]
sc2 = ax.scatter(Diff_isomap_RPPA[:, 0], Diff_isomap_RPPA[:, 1],
                 c="blue", marker="x", s=20, alpha=0.6, label="Diffusion (ISOMAP RPPA)")
# sc3 = ax.scatter(Diff_RPPA[:, 1], Diff_RPPA[:, 2],
#                  c="blue", marker="o", s=20, alpha=0.6, label="Diffusion (Original z-scored RPPA)")

# Number labels
for i in range(len(X_rppa)):
    ax.text(Diff_isomap_RPPA[i, 0], Diff_isomap_RPPA[i, 1], str(i),
            fontsize=6, alpha = 0.6, ha='right', va='bottom')
    # ax.text(Diff_RPPA[i, 0], Diff_RPPA[i, 1], str(i),
    #         fontsize=6, alpha = 0.6, ha='left', va='bottom')

ax.set_title(f"Diffusion Map Comparison (t = {t})")
ax.set_aspect('equal')
ax.grid(alpha=0.3)
ax.legend(loc='best')   # ✅ Add legend here to distinguish datasets

plt.tight_layout()
plt.show()

In [ ]:
# Diffusion Map of ISOMAP RPPA data and the original RPPA data


# Fit
model_isomap_rna = diffusion_map_fit(isomap_embedding_rna, n_components=2)
model_rna = diffusion_map_fit(X_rna, n_components=2)
# --- Plot diffusion maps of ISOMAP RPPA and original RPPA data ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ===== (1) Left panel: ISOMAP RPPA embedding =====
ax = axes[0]
sc1 = ax.scatter(isomap_embedding_rna[:, 0], isomap_embedding_rna[:, 1],
                 c="red", marker="x", s=20, alpha=0.6, label="ISOMAP RNA")
for i in range(len(X_rna)):
    ax.text(isomap_embedding_rna[i, 0], isomap_embedding_rna[i, 1],
            str(i), fontsize=8, ha='right', va='bottom')

ax.set_title("ISOMAP RNA Embedding")
ax.set_aspect('equal')
ax.grid(alpha=0.3)
ax.legend(loc='best')

# ===== (2) Right panel: Diffusion Map comparison =====
t = 0.01
Diff_isomap_RNA = diffusion_map_transform(model_isomap_rna, t)
Diff_RNA = diffusion_map_transform(model_rna, t)

ax = axes[1]
sc2 = ax.scatter(Diff_isomap_RNA[:, 0], Diff_isomap_RNA[:, 1],
                 c="red", marker="x", s=20, alpha=0.6, label="Diffusion (ISOMAP RNA)")
# sc3 = ax.scatter(Diff_RNA[:, 0], Diff_RNA[:, 1],
#                  c="blue", marker="o", s=20, alpha=0.6, label="Diffusion (Original z-scored RNA)")

# Number labels
for i in range(len(X_rna)):
    ax.text(Diff_isomap_RNA[i, 0], Diff_isomap_RNA[i, 1], str(i),
            fontsize=6, alpha = 0.6, ha='right', va='bottom')
#     ax.text(Diff_RNA[i, 0], Diff_RNA[i, 1], str(i),
#             fontsize=6, alpha = 0.6, ha='left', va='bottom')

ax.set_title(f"Diffusion Map Comparison (t = {t})")
ax.set_aspect('equal')
ax.grid(alpha=0.3)
ax.legend(loc='best')  

plt.tight_layout()
plt.show()

In [ ]:
pip install umap-learn

In [ ]:
# visualize RPPA and RNA by UMAP
import umap

# UMAP on RPPA data
umap_rppa = umap.UMAP(n_components=2, random_state=0)
umap_embedding_rppa = umap_rppa.fit_transform(X_rppa)
print("The shape of umap_embedding_rppa is:", umap_embedding_rppa.shape)

# UMAP on RNA data
umap_rna = umap.UMAP(n_components=2, random_state=0)
umap_embedding_rna = umap_rna.fit_transform(X_rna)
print("The shape of umap_embedding_rna is:", umap_embedding_rna.shape)

# Plot UMAP embeddings side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# RPPA UMAP
axes[0].scatter(umap_embedding_rppa[:, 0], umap_embedding_rppa[:, 1], c="blue", cmap='Spectral', alpha=0.7)
axes[0].set_title("RPPA UMAP Embedding")
axes[0].set_xlabel("UMAP1")
axes[0].set_ylabel("UMAP2")

# Number labels
for i in range(len(X_rppa)):
    axes[0].text(umap_embedding_rppa[i, 0], umap_embedding_rppa[i, 1],
            str(i), fontsize=10, ha='right', va='bottom')
# RNA UMAP
axes[1].scatter(umap_embedding_rna[:, 0], umap_embedding_rna[:, 1], c="red", cmap='Spectral', alpha=0.7)
axes[1].set_title("RNA UMAP Embedding")
axes[1].set_xlabel("UMAP1")
axes[1].set_ylabel("UMAP2")

# Number labels
for i in range(len(X_rna)):
    axes[1].text(umap_embedding_rna[i, 0], umap_embedding_rna[i, 1],
            str(i), fontsize=10, ha='right', va='bottom')
plt.tight_layout()
plt.show()



In [ ]:
# matched line plot for UMAP embeddings between RPPA and RuntimeWarning

plt.figure(figsize=(15, 18))
plt.scatter(umap_embedding_rppa[:, 0], umap_embedding_rppa[:, 1], c="blue", alpha=0.6, label='RPPA UMAP')
plt.scatter(umap_embedding_rna[:, 0], umap_embedding_rna[:, 1], c="red", marker='x', alpha=0.8, label='RNA UMAP')
for i in range(0, len(X_rppa), 1):
    plt.plot([umap_embedding_rppa[i, 0], umap_embedding_rna[i, 0]],
             [umap_embedding_rppa[i, 1], umap_embedding_rna[i, 1]],
             color='gray', alpha=0.3, linewidth=0.8)
    
# Number labels
for i in range(len(X_rppa)):
    plt.text(umap_embedding_rna[i, 0], umap_embedding_rna[i, 1],
             str(i), fontsize=10, ha='right', va='bottom')
    plt.text(umap_embedding_rppa[i, 0], umap_embedding_rppa[i, 1],
             str(i), fontsize=10, ha='right', va='bottom')
plt.title("Matched Line Plot: RPPA UMAP vs RNA UMAP")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend()
plt.gca().set_aspect('equal', adjustable='box')
plt.show()

In [ ]:
# matched line plot for ISOMAP embeddings between RPPA and RNA
plt.figure(figsize=(15, 18))
plt.scatter(isomap_embedding_rppa[:, 0], isomap_embedding_rppa[:, 1], c="blue", alpha=0.6, label='RPPA ISOMAP')
plt.scatter(isomap_embedding_rna[:, 0], isomap_embedding_rna[:, 1], c="red", marker='x', alpha=0.8, label='RNA ISOMAP')
for i in range(0, len(X_rppa), 1):
    plt.plot([isomap_embedding_rppa[i, 0], isomap_embedding_rna[i, 0]],
             [isomap_embedding_rppa[i, 1], isomap_embedding_rna[i, 1]],
             color='gray', alpha=0.3, linewidth=0.8)
# Number labels
for i in range(len(X_rppa)):
    plt.text(isomap_embedding_rna[i, 0], isomap_embedding_rna[i, 1],
             str(i), fontsize=10, ha='right', va='bottom')
    plt.text(isomap_embedding_rppa[i, 0], isomap_embedding_rppa[i, 1],
             str(i), fontsize=10, ha='right', va='bottom')
plt.title("Matched Line Plot: RPPA ISOMAP vs RNA ISOMAP")
plt.xlabel("ISOMAP1")
plt.ylabel("ISOMAP2")
plt.legend()
plt.gca().set_aspect('equal', adjustable='box')
plt.show()

In the following section, we will load the clinical data that contains survival times of the patients. The loaded file is "data_clinical_patient.txt". We focus on the analysis of the survivial time. 

In [ ]:
# Clinical data (time, event)
clin = pd.read_csv(clinFilePath, sep="\t", comment="#")
print("Shape of clinical data:", clin.shape)

clin_sub = clin[clin["PATIENT_ID"].isin(common_samples)].copy()
print("Shape of clinical data in common_genes rows:", clin_sub.shape)
# clin_common = clin.loc[common_samples]

In [ ]:
common_samples[0]

In [ ]:
subset = clin[clin["PATIENT_ID"].isin([x[:12] for x in common_samples])]
print("the shape of subset:", subset.shape)
subset.head()

In [ ]:
# clin[clin["PATIENT_ID"] == "TCGA-02-2485"]
subset[["OS_STATUS", "OS_MONTHS"]].isna().sum()

In [ ]:
print(subset.isna().sum())

no_nan_cols = subset.columns[subset.notna().all()]
print("Columns without NaN values:", no_nan_cols)
subset_no_nan = subset[no_nan_cols]
subset_no_nan.head(10)

| Column                     | Meaning                       | Biological/Clinical Use           |
| -------------------------- | ----------------------------- | --------------------------------- |
| `PATIENT_ID`               | Unique identifier             | Used for merging data             |
| `CANCER_TYPE_ACRONYM`      | Tumor type code               | Distinguishes cancer cohorts      |
| `IN_PANCANPATHWAYS_FREEZE` | Inclusion flag                | QC for curated PanCancer datasets |
| `OS_STATUS`                | Alive or deceased             | Overall survival endpoint         |
| `OS_MONTHS`                | Survival time                 | Quantitative OS measure           |
| `DSS_MONTHS`               | Cancer-specific survival time | Focused on tumor-caused deaths    |
| `PFS_STATUS`               | Progressed or not             | Disease progression endpoint      |
| `PFS_MONTHS`               | Time to progression           | Tumor control duration            |
| `GENETIC_ANCESTRY_LABEL`   | Germline ancestry group       | Population stratification         |



In [ ]:
subset_no_nan.describe()

In [ ]:
X_rna[:10]

In [ ]:
# X_rppa = R_z.T.values
# X_rna = rna_hv_z.values

R_z.head() 

Rppa_z = R_z.copy()
Rppa_z.columns = [c[:12] for c in R_z.columns]
Rppa_z.head()

# count how many times each patient ID appears in the columns
counts = pd.Series(Rppa_z.columns).value_counts()
duplicates = counts[counts > 1]
print("Patients with multiple samples:", len(duplicates))
print(duplicates.head())

Rppa_z = Rppa_z.groupby(axis =  1, level = 0).mean()
counts_after = pd.Series(Rppa_z.columns).value_counts()
duplicates_after = counts_after[counts_after > 1]
print("Patients with multiple samples after averaging:", len(duplicates_after))
# Double-check uniqueness
print("Total columns:", len(Rppa_z.columns))
print("Unique patient IDs:", len(set(Rppa_z.columns)))
Rppa_z.head()

In [ ]:
# subset_no_nan must have PATIENT_ID, OS_MONTHS, OS_STATUS (or OS_EVENT)
clin = subset_no_nan.copy()
clin.columns = [c.upper() for c in clin.columns]

# If OS_EVENT not built yet:
if "OS_EVENT" not in clin.columns:
    clin["OS_EVENT"] = clin["OS_STATUS"].str.upper().map(
        lambda s: 1 if isinstance(s,str) and ("DECEASED" in s or s.startswith("1")) else 0
    )

# Keep only needed columns
clin = clin[["PATIENT_ID","OS_MONTHS","OS_EVENT"]].dropna(subset=["OS_MONTHS","OS_EVENT"])
clin["OS_MONTHS"] = pd.to_numeric(clin["OS_MONTHS"], errors="coerce")
clin.head()

In [ ]:
# Rppa_z: rows = proteins, columns = 12-char patient IDs (already cleaned)
common_patients = pd.Index(Rppa_z.columns).intersection(clin["PATIENT_ID"])

# Subset & orient: rows = patients, cols = proteins
rppa_mat = Rppa_z.loc[:, common_patients].T

rppa_mat.head()


In [ ]:


# Clinical aligned to same order
clin_rppa = clin.set_index("PATIENT_ID").loc[common_patients] # align clinical to rppa patients
df_rppa = pd.concat([clin_rppa[["OS_MONTHS","OS_EVENT"]], rppa_mat], axis=1).dropna(subset=["OS_MONTHS","OS_EVENT"])
print("RPPA Cox matrix:", df_rppa.shape)  # (n_patients, 2 + n_proteins)




In [ ]:
# RNA
Rna_z = rna_hv_z.copy()
print("The shape of Rna_z before changing columns:", Rna_z.shape)
Rna_z.head()
Rna_z.index = [c[:12] for c in Rna_z.index]
print(Rna_z.head())

# Collapse duplicate patient IDs by averaging
Rna_z = Rna_z.groupby(level = 0).mean()
common_patients_rna = pd.Index(Rna_z.index).intersection(subset_no_nan["PATIENT_ID"])


In [ ]:

Rna_sub = Rna_z.loc[common_patients_rna]
clin_rna = clin.set_index("PATIENT_ID").loc[common_patients_rna]
print(clin_rna.columns)

In [ ]:

# merge into one dataframe
df_rna = pd.concat([clin_rna[["OS_MONTHS","OS_EVENT"]], Rna_sub], axis=1)
print("RNA Cox matrix:", df_rna.shape)




In [ ]:
#Get lifelines for Cox analysis
!pip install lifelines

In [ ]:
from lifelines import CoxPHFitter
from sklearn.decomposition import PCA

# Using direct features for Cox regression (RNA)
# cph = CoxPHFitter(penalizer=1)
# cph.fit(df_rna, duration_col="OS_MONTHS", event_col="OS_EVENT")
# cph.print_summary()


In [ ]:
# Using PCA components for Cox regression 
X_rna = df_rna.drop(columns=["OS_MONTHS","OS_EVENT"])
pca = PCA(n_components=10)
X_rna_pca = pd.DataFrame(
    pca.fit_transform(X_rna),
    index=X_rna.index,
    columns=[f"PC{i+1}" for i in range(10)]
)

df_rna_pca = pd.concat([df_rna[["OS_MONTHS","OS_EVENT"]], X_rna_pca], axis=1)

cph = CoxPHFitter()
cph.fit(df_rna_pca, duration_col="OS_MONTHS", event_col="OS_EVENT")
cph.print_summary()

From this Cox analysis of Top 10 Principal Component Embedding of RNA data, we see that PC2 and PC8 has the most significant influence on the OS_MONTHs since the p-value of PC2 is only 0.0096 and PC8 is only 0.047, both < 0.05.


In [ ]:
df_rna_pca.head()

In [ ]:
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

def km_plot_median(df, feature, ax=None, time_col="OS_MONTHS", event_col="OS_EVENT", title=None):
    d = df[[time_col, event_col, feature]].dropna()
    med = d[feature].median()
    high = d[d[feature] > med]
    low  = d[d[feature] <= med]

    kmh, kml = KaplanMeierFitter(), KaplanMeierFitter()
    kmh.fit(high[time_col], high[event_col], label="High")
    kml.fit(low[time_col],  low[event_col],  label="Low")

    lr = logrank_test(high[time_col], low[time_col], high[event_col], low[event_col])
    pval = lr.p_value

    if ax is None:
        ax = plt.gca()

    kmh.plot_survival_function(ax=ax, ci_show=False)
    kml.plot_survival_function(ax=ax, ci_show=False)
    ax.set_xlabel("Months")
    ax.set_ylabel("Survival probability")
    ax.set_title(title or f"{feature} (p={pval:.3g})")
    ax.grid(alpha=0.3)

    return pval


# ==== Plot 10 PCs in a 4x3 grid ====
fig, axes = plt.subplots(4, 3, figsize=(14, 10))
axes = axes.flatten()

for i in range(10):  # PC1 ... PC10
    km_plot_median(df_rna_pca, feature=f"PC{i+1}", ax=axes[i])

# Hide extra empty subplots
for j in range(10, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
rppa_mat
clin_rppa[["OS_MONTHS","OS_EVENT"]]
# df_rppa["OS_EVENT"].dtype

print("check NaNs in df:",df_rppa.isna().sum().sum())
df_rppa.describe()

In [ ]:
from lifelines import CoxPHFitter
cph = CoxPHFitter(penalizer=5)
cph.fit(df_rppa, duration_col="OS_MONTHS", event_col="OS_EVENT")
cph.print_summary()

From this Cox analysis of Top 10 Principal Component Embedding of RPPA data, we did not find any features with significant influence on the OS_MONTHs. All principal components have large p-values. 


In [ ]:
X_rppa = df_rppa.drop(columns=["OS_MONTHS","OS_EVENT"])
pca = PCA(n_components=10)  # top 10 PCs
X_rppa_pca = pd.DataFrame(pca.fit_transform(X_rppa), index=X_rppa.index, columns=[f"PC{i+1}" for i in range(10)])

df_rppa_pca = pd.concat([df_rppa[["OS_MONTHS","OS_EVENT"]], X_rppa_pca], axis=1)
cph.fit(df_rppa_pca, duration_col="OS_MONTHS", event_col="OS_EVENT")
cph.print_summary()


In [ ]:
def km_plot_median(df, feature, ax=None, time_col="OS_MONTHS", event_col="OS_EVENT", title=None):
    d = df[[time_col, event_col, feature]].dropna()
    med = d[feature].median()
    high = d[d[feature] > med]
    low  = d[d[feature] <= med]

    kmh, kml = KaplanMeierFitter(), KaplanMeierFitter()
    kmh.fit(high[time_col], high[event_col], label="High")
    kml.fit(low[time_col],  low[event_col],  label="Low")

    lr = logrank_test(high[time_col], low[time_col], high[event_col], low[event_col])
    pval = lr.p_value

    if ax is None:
        ax = plt.gca()
    kmh.plot_survival_function(ax=ax, ci_show=False)
    kml.plot_survival_function(ax=ax, ci_show=False)
    ax.set_xlabel("Months"); ax.set_ylabel("Survival probability")
    ax.set_title(title or f"{feature} (p={pval:.3g})")
    ax.grid(alpha=0.3)
    return pval

# -------- choose 10 RPPA features to plot --------
# Option A: top-variance proteins (simple, no screening step)
rppa_feats_all = df_rppa.columns.drop(["OS_MONTHS","OS_EVENT"])
top10_rppa = (
    df_rppa[rppa_feats_all]
    .var(axis=0)
    .sort_values(ascending=False)
    .head(10)
    .index.tolist()
)

# (Optional) Option B: if you already ran univariate Cox and have rppa_results with a 'feature' and 'p' column:
# top10_rppa = rppa_results.sort_values("p").head(10)["feature"].tolist()

# -------- plot 10 KM curves in a 4x3 grid --------
fig, axes = plt.subplots(4, 3, figsize=(14, 10))
axes = axes.flatten()

for i, feat in enumerate(top10_rppa):
    km_plot_median(df_rppa, feature=feat, ax=axes[i])

# Hide any leftover empty subplots (there will be 2 empty spots)
for j in range(len(top10_rppa), len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()
